In [2]:
from bsm2_python.bsm2_ol import BSM2OL
from bsm2_python.bsm2.plantperformance import PlantPerformance
import bsm2_python.bsm2.init.plantperformanceinit_bsm2 as pp_init
from bsm2_python.bsm2.init.aerationcontrolinit import KLA3GAIN, KLA5GAIN
from bsm2_python.gases.gases import GasMix
from bsm2_python.bsm2.init.reginit_bsm2 import T_OP
from tqdm import tqdm
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pickle as pkl
import numpy as np
from nn_functions import choose_kla_4, load_scaler_and_pca, save_scaler_and_pca,transform_influent,fit_influent_scaler_and_pca




19:36:06.189 INFO     Note: NumExpr detected 16 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
Note: NumExpr detected 16 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
19:36:06.191 INFO     NumExpr defaulting to 8 threads.
NumExpr defaulting to 8 threads.


In [5]:

# Initialize the influent scaler and PCA
SI, SS, XI, XS, XBH, XBA, XP, SO, SNO, SNH, SND, XND, SALK, TSS, Q, TEMP, SD1, SD2, SD3, XD4, XD5 = np.arange(21)
bsm2 = BSM2OL()
pp = PlantPerformance(pp_init.PP_PAR)
scaler_influent, pca_influent = fit_influent_scaler_and_pca(bsm2.y_in, n_components=5)
save_scaler_and_pca(scaler_influent, pca_influent, 'influent_scaler_pca_params.pkl')

# Setting up parameters
last_steps = 144
select = np.array([0, 60, 120, 180, 240])
num_executions = 1
num_features = len((0, *range(pca_influent.n_components)))
num_targets = 7  # Number of effluent, biogas_heat, and heat_demand components

feature_container = np.zeros((0, num_features * last_steps))
target_container = np.zeros((0, num_targets))

# Make the Generation loop 
for num_execution in range(num_executions):
    bsm2 = BSM2OL()
    bsm2.stabilize()
    kla_4 = 0
    effluent = np.zeros(5)
    biogas_heat = 0
    heat_demand = 0
    write_cols = len((kla_4, *range(pca_influent.n_components), *effluent, biogas_heat, heat_demand))
    data = np.zeros((bsm2.simtime.size, write_cols))
    features = np.zeros((bsm2.simtime.size - last_steps, num_features * last_steps))
    targets = np.zeros((bsm2.simtime.size - last_steps, num_targets))

    for idx, step in enumerate(tqdm(bsm2.simtime)):
        kla_4 = choose_kla_4(select, num_execution, kla_4)
        klas = np.array([0, 0, KLA3GAIN * kla_4, kla_4, KLA5GAIN * kla_4])
        
        # Get influent data and apply combined scaling & PCA transformation
        influent_data = bsm2.y_in[np.where(bsm2.data_time <= step)[0][-1], :].reshape(1, -1)
        influent_transformed = transform_influent(influent_data, scaler_influent, pca_influent).flatten()
        
        bsm2.step(idx, klas)
        
        # Calculate effluent data and other metrics
        adv_eff = pp.advanced_quantities(bsm2.y_eff, components=('totalN', 'COD', 'BOD5'))
        tot_n, cod, bod5 = adv_eff[0, :]
        effluent = np.array((bsm2.y_eff[SNH], bsm2.y_eff[TSS], tot_n, cod, bod5))
        
        ch4_prod, h2_prod, co2_prod, q_gas = pp.gas_production(bsm2.yd_out, T_OP, p_atm=1.0130)
        biogas = GasMix(ch4_prod, co2_prod, h2_prod)
        biogas_heat = biogas.h_u * q_gas / 24  # in kW
        heat_demand = pp.heat_demand_step(bsm2.yd_in, T_OP)  # in kW
        
        # Save the combined data
        data[idx, :] = np.hstack((kla_4, influent_transformed, effluent, biogas_heat, heat_demand))
        
        if idx >= last_steps:
            features[idx-last_steps] = data[idx-last_steps:idx, :num_features].flatten()
            targets[idx-last_steps] = np.array((*effluent, biogas_heat, *heat_demand))
    feature_container = np.concatenate((feature_container, features), axis=0)
    target_container = np.concatenate((target_container, targets), axis=0)


19:55:27.651 INFO     Stabilized after 5624 iterations

Stabilized after 5624 iterations



100%|██████████| 58465/58465 [07:56<00:00, 122.61it/s]


In [6]:
# save data to .npz file
np.savez('data_bsm2_pca80.npz', features=feature_container, targets=target_container)


### Data with 11 influent components

In [40]:

from bsm2_python.bsm2_ol import BSM2OL
from bsm2_python.bsm2.plantperformance import PlantPerformance
import bsm2_python.bsm2.init.plantperformanceinit_bsm2 as pp_init
from bsm2_python.bsm2.init.aerationcontrolinit import KLA3GAIN, KLA5GAIN
from bsm2_python.gases.gases import GasMix
from bsm2_python.bsm2.init.reginit_bsm2 import T_OP
from tqdm import tqdm
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle as pkl
from nn_functions import choose_kla_4

# Constants
SI, SS, XI, XS, XBH, XBA, XP, SO, SNO, SNH, SND, XND, SALK, TSS, Q, TEMP, SD1, SD2, SD3, XD4, XD5 = np.arange(21)
red_dims = [SI, SS, XI, XS, XBH, SNH, SND, XND, TSS, Q, TEMP]

# Initialize PlantPerformance
pp = PlantPerformance(pp_init.PP_PAR)


# Function to reduce dimensionality
def reduce_dimensionality(y_in):
    return y_in[red_dims]

# Generate a new array with input data from the past 36 hours (144 steps) for each timestep and the corresponding output data
last_steps = 144
bsm2 = BSM2OL()

# Do the simulation loop
# Let the simulator select between 0, 60, 120, 180, and 240 1/h for kla_4
select = np.array([0, 60, 120, 180, 240])
num_executions = 1  

num_features = 12  
num_targets = 7  
feature_container = np.zeros((0, num_features * last_steps))
target_container = np.zeros((0, num_targets))

for num_execution in range(num_executions):
    bsm2 = BSM2OL()
    bsm2.stabilize()
    effluent = 5 * [0]
    biogas_heat = 0
    heat_demand = 0
    # Setup the container array
    write_cols = len((0, *range(num_features), *effluent, biogas_heat, heat_demand))
    data = np.zeros((bsm2.simtime.size, write_cols))
    features = np.zeros((bsm2.simtime.size - last_steps, num_features * last_steps))
    targets = np.zeros((bsm2.simtime.size - last_steps, num_targets))
    
    for idx, step in enumerate(tqdm(bsm2.simtime)):
        # Randomly select kla_4
        kla_4 = np.random.choice(select)
        klas = np.array([0, 0, KLA3GAIN * kla_4, kla_4, KLA5GAIN * kla_4])
        
        influent = bsm2.y_in[np.where(bsm2.data_time <= step)[0][-1], :]
        influent_reduced = reduce_dimensionality(influent).reshape(1, -1)
        
        # Scale the reduced influent data
        #influent_scaled = scaler.transform(influent_reduced).flatten()
        
        bsm2.step(idx, klas)
        
        # Calculate special variables ntot, cod, bod5
        adv_eff = pp.advanced_quantities(bsm2.y_eff, components=('totalN', 'COD', 'BOD5'))
        tot_n, cod, bod5 = adv_eff[0, :]
        
        effluent = np.array((bsm2.y_eff[SNH], bsm2.y_eff[TSS], tot_n, cod, bod5))
        
        ch4_prod, h2_prod, co2_prod, q_gas = pp.gas_production(bsm2.yd_out, T_OP, p_atm=1.0130)
        # Calculate the heating value yourself (in kW please)
        biogas = GasMix(ch4_prod, co2_prod, h2_prod)
        biogas_heat = biogas.h_u * q_gas / 24  # kW
        
        # Get the heat demand
        heat_demand = pp.heat_demand_step(bsm2.yd_in, T_OP)  # kW
        
        # Now save the data to the data acquisition
        data[idx, :] = np.hstack((kla_4, influent_reduced.flatten(), effluent, biogas_heat, heat_demand[:2], 0))  # Ensure the shape matches
        
        if idx >= last_steps:
            features[idx - last_steps] = data[idx - last_steps:idx, :num_features].flatten()
            targets[idx - last_steps] = np.array((*effluent, biogas_heat, *heat_demand[:2]))
    
    feature_container = np.concatenate((feature_container, features), axis=0)
    target_container = np.concatenate((target_container, targets), axis=0)

# Generate the diverse data with many probabilities of 50,60,70,80,90,100 



21:19:56.129 INFO     Stabilized after 5622 iterations

Stabilized after 5622 iterations



100%|██████████| 58465/58465 [07:45<00:00, 125.68it/s]


In [41]:
np.savez('data_bsm2_1728_50.npz', features=feature_container, targets=target_container)


In [4]:
import pandas as pd
# Convert to DataFrame for sampling
features_df = pd.DataFrame(feature_container)
targets_df = pd.DataFrame(target_container)

# Sample n rows
sampled_features = features_df.sample(n=1000, replace=True, random_state=42)
sampled_targets = targets_df.loc[sampled_features.index]  # Ensure targets match sampled features

# Save the sampled test data
np.savez(r'C:\Users\AMEEN\bsm-2-python-lin-opt\Updated Progress files\test_data_bsm2_1728.npz', 
         features=sampled_features.values, targets=sampled_targets.values)

print("Test data saved successfully.")


Test data saved successfully.


In [10]:
import numpy as np

# Load each dataset
data1_50 = np.load(r'C:\Users\AMEEN\bsm-2-python-lin-opt\Updated Progress files\data_bsm2_pca50.npz')
data1_60 = np.load(r'C:\Users\AMEEN\bsm-2-python-lin-opt\Updated Progress files\data_bsm2_pca60.npz')
data1_70 = np.load(r'C:\Users\AMEEN\bsm-2-python-lin-opt\Updated Progress files\data_bsm2_pca70.npz')
data1_80 = np.load(r'C:\Users\AMEEN\bsm-2-python-lin-opt\Updated Progress files\data_bsm2_pca80.npz')

# Print keys to inspect the correct key names
print("Keys in data1_50:", data1_50.keys())
print("Keys in data1_60:", data1_60.keys())
print("Keys in data1_70:", data1_70.keys())
print("Keys in data1_80:", data1_80.keys())

# Assuming each file contains arrays under specific keys, e.g., 'features' and 'targets'
# Extract each array from the files
array_50 = data1_50['targets']
array_60 = data1_60['targets']
array_70 = data1_70['targets']
array_80 = data1_80['targets']

# Combine the arrays along a specific axis, e.g., rows (axis=0)
combined_data_target = np.concatenate((array_50, array_60, array_70, array_80), axis=0)

# Now combined_data holds the concatenated array


Keys in data1_50: KeysView(NpzFile 'C:\\Users\\AMEEN\\bsm-2-python-lin-opt\\Updated Progress files\\data_bsm2_pca50.npz' with keys: features, targets)
Keys in data1_60: KeysView(NpzFile 'C:\\Users\\AMEEN\\bsm-2-python-lin-opt\\Updated Progress files\\data_bsm2_pca60.npz' with keys: features, targets)
Keys in data1_70: KeysView(NpzFile 'C:\\Users\\AMEEN\\bsm-2-python-lin-opt\\Updated Progress files\\data_bsm2_pca70.npz' with keys: features, targets)
Keys in data1_80: KeysView(NpzFile 'C:\\Users\\AMEEN\\bsm-2-python-lin-opt\\Updated Progress files\\data_bsm2_pca80.npz' with keys: features, targets)


In [ ]:
np.savez('data_bsm2_pca_smooth_aggressive_patterns.npz', features=feature_container, targets=target_container)
